### Импорты и создание модели

In [1]:
import sys
sys.path.append('../src')

import torch
from pathlib import Path
import pandas as pd

from dataset import build_item_vocab, build_user_sequences
from model import create_model, recommend_top_k

PROC = Path('../data/processed')
train = pd.read_parquet(PROC / 'train.parquet')

item2idx, idx2item = build_item_vocab(train)
vocab_size = len(item2idx) + 2
print(f"Vocab size: {vocab_size}")

model = create_model(vocab_size=vocab_size)
print(f"Модель создана: {type(model).__name__}")

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 14936), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 14936), got 50256. This may result in unexpected behavior.


Vocab size: 14937
Модель создана: GPT2LMHeadModel


### Кол-во параметров у модели

In [2]:
# Общее число параметров
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Всего параметров:       {total_params:,}")
print(f"Обучаемых параметров:   {trainable_params:,}")
print(f"Размер в fp32:          {total_params * 4 / 1024**2:.1f} MB")

# Разбивка по частям
print("\nПо компонентам:")
for name, param in model.named_parameters():
    if 'wte' in name:  # word token embeddings
        print(f"  Token embeddings:    {param.numel():,} ({name})")
    elif 'wpe' in name:  # word position embeddings
        print(f"  Position embeddings: {param.numel():,} ({name})")

Всего параметров:       2,315,136
Обучаемых параметров:   2,315,136
Размер в fp32:          8.8 MB

По компонентам:
  Token embeddings:    1,911,936 (transformer.wte.weight)
  Position embeddings: 6,400 (transformer.wpe.weight)


### Forward pass

In [3]:
# Для начала - одна последовательность из реального train
sequences = build_user_sequences(train, max_len=50)

# Поиск юзера с 5+ айтемами
sample_user = None
for uid, seq in sequences.items():
    if len(seq) >= 5:
        sample_user = uid
        sample_seq = seq
        break

print(f"Юзер {sample_user}: {len(sample_seq)} айтемов")
print(f"История (item_id): {sample_seq}")

# Перевод в индексы
indexed = [item2idx[i] for i in sample_seq]
print(f"История (индексы): {indexed}")

# Паддинг слева до длины 50
pad_len = 50 - len(indexed)
input_ids = torch.tensor([0] * pad_len + indexed, dtype=torch.long).unsqueeze(0)
print(f"input_ids.shape: {input_ids.shape}")

# Прогон
model.eval()
with torch.no_grad():
    outputs = model(input_ids=input_ids)
    logits = outputs.logits

print(f"logits.shape: {logits.shape}")
print(f"  -> batch_size=1, seq_len=50, vocab_size={vocab_size}")

# Логиты для последней позиции (то, что модель предсказывает после всей истории)
last_logits = logits[0, -1, :]
print(f"last_logits.shape: {last_logits.shape}")

# Топ-5 предсказаний
top_5 = torch.topk(last_logits, 5)
print(f"\nТоп-5 предсказаний (индексы в словаре):")
for idx, logit in zip(top_5.indices.tolist(), top_5.values.tolist()):
    print(f"  index {idx} (item_id={idx2item.get(idx, 'PAD/UNK')}): logit={logit:.3f}")

Юзер 0: 6 айтемов
История (item_id): [12192, 7102, 14359, 15297, 9728, 6006]
История (индексы): [10979, 6394, 12959, 13823, 8771, 5406]
input_ids.shape: torch.Size([1, 50])
logits.shape: torch.Size([1, 50, 14937])
  -> batch_size=1, seq_len=50, vocab_size=14937
last_logits.shape: torch.Size([14937])

Топ-5 предсказаний (индексы в словаре):
  index 5406 (item_id=6006): logit=1.173
  index 12303 (item_id=13637): logit=0.872
  index 5551 (item_id=6165): logit=0.836
  index 8986 (item_id=9966): logit=0.815
  index 11133 (item_id=12365): logit=0.797


*Модель пока не обучена, поэтому результаты случайны. Данная проверка не ставила перед собой цели узнать точнось модели - просто проверка, что она работает в целом*

### Тест recommend_top_k

In [4]:
# Написанная вспомогательная функцию
recommendations = recommend_top_k(model, input_ids, k=10)
print(f"Top-10 рекомендаций (индексы): {recommendations}")

# Проверка, что PAD не попал
assert 0 not in recommendations, "PAD не должен быть в рекомендациях"
print("PAD не в топе")

# Проверим, что функция не изменила модель
assert model.training == False, "recommend_top_k должен перевести модель в eval"
print("Модель в eval-режиме")

# Проверка форму контекста (1, seq_len) и (seq_len,)
recs_1d = recommend_top_k(model, input_ids.squeeze(0), k=10)
recs_2d = recommend_top_k(model, input_ids, k=10)
assert recs_1d == recs_2d, "Функция должна работать одинаково с 1D и 2D контекстом"
print("recommend_top_k работает с 1D и 2D")

Top-10 рекомендаций (индексы): [5406, 12303, 5551, 8986, 11133, 3741, 12296, 9023, 2469, 2026]
PAD не в топе
Модель в eval-режиме
recommend_top_k работает с 1D и 2D


*Проблем нет, всё как и планировалось, модель прошла все проверки и не показала ни одного бага.*